## Setup

Loads the libraries, folders, version labels, random seed, and shared constants used by the rest of the notebook.

In [1]:
import hashlib
import json
import os
import pickle
import re
import time
import uuid
from datetime import UTC, datetime
from functools import lru_cache
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import bm25s
import numpy as np
import pandas as pd
import stopwordsiso as stopwords
from dotenv import load_dotenv
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.metrics import ndcg_score
from sklearn.metrics.pairwise import haversine_distances, linear_kernel
from sklearn.preprocessing import minmax_scale

DATASET_VERSION = "v1"
MODEL_VERSION = "hybrid_bm25_dense_geo_v1"

ROOT_DIR = Path.cwd().parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
INDEX_DIR = ROOT_DIR / "data" / "indexes"
BENCHMARK_DIR = ROOT_DIR / "data" / "benchmarks"
AUDIT_DIR = ROOT_DIR / "data" / "audit"

load_dotenv(ROOT_DIR / ".env")

for directory in [PROCESSED_DIR, INDEX_DIR, BENCHMARK_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TOP_K = 10
RANDOM_SEED = 31

pd.set_option("display.max_colwidth", 250)

/Users/riccardo/Projects/luiss-xai/.venv/lib/python3.12/site-packages/stopwordsiso/_core.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Schema validation

Checks the raw country CSV files before any indexes are built. Each file must contain the fields used later for retrieval, geography, budget benchmarks, and result display.

In [2]:
REQUIRED_COLUMNS = [
    "Operation_Name_English",
    "Operation_Summary_English",
    "Operation_Name_Programme_Language",
    "Operation_Summary_Programme_Language",
    "Country",
    "CountryCode",
    "Location_Indicator_latitude_longitude",
    "NUTS3_Label",
    "LAU_Labels",
    "Operation_Unique_Identifier",
    "Programme_Name",
    "Fund_Name",
    "Category_Label",
    "Specific_Objective_Label",
    "Policy_Objective_Label",
    "Total_Eligible_Expenditure_amount",
    "Project_EU_Budget",
]

csv_files = sorted(RAW_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_DIR}")

schema_report = []
for csv_file in csv_files:
    header = pd.read_csv(csv_file, nrows=0).columns.tolist()
    missing = sorted(set(REQUIRED_COLUMNS) - set(header))
    schema_report.append(
        {
            "file": csv_file.name,
            "column_count": len(header),
            "missing_required_columns": missing,
        }
    )

schema_df = pd.DataFrame(schema_report)
files_with_issues = schema_df[schema_df["missing_required_columns"].str.len() > 0]

if files_with_issues.empty:
    print("All files contain the required columns.")
else:
    display(files_with_issues)

All files contain the required columns.


## Project corpus

Builds the searchable project corpus. It cleans text, standardizes labels and country codes, parses budgets, extracts coordinates, and combines English and programme-language names and summaries into `search_text`. Dense retrieval, the hybrid model, and the explanation layer all use this readable field.

The processed dataset contains 124,988 records from 26 countries. The sample row shows both the output fields and the text that will be searched.

In [3]:
def clean_text(value):
    if pd.isna(value):
        return ""
    # collapse any number of whitespace into one space
    return re.sub(r"\s+", " ", str(value)).strip()


def clean_label(value):
    text = clean_text(value)
    return text if text else np.nan


PROCESSED_COLUMNS = [
    "record_id",
    "Operation_Unique_Identifier",
    "Programme_Name",
    "Fund_Name",
    "Category_Label",
    "Specific_Objective_Label",
    "Policy_Objective_Label",
    "Total_Eligible_Expenditure_amount",
    "Project_EU_Budget",
    "Country",
    "CountryCode",
    "NUTS3_Label",
    "LAU_Labels",
    "latitude",
    "longitude",
    "search_text",
    "dataset_version",
]
processed_path = PROCESSED_DIR / f"eu_projects_{DATASET_VERSION}.parquet"

if processed_path.exists():
    processed_df = pd.read_parquet(processed_path)
else:
    frames = []
    for csv_file in csv_files:
        frames.append(pd.read_csv(csv_file, usecols=REQUIRED_COLUMNS, low_memory=False))

    raw_df = pd.concat(frames, ignore_index=True)

    for column in raw_df.select_dtypes(include=["object", "str"]).columns:
        raw_df[column] = raw_df[column].map(clean_text)

    processed_df = raw_df.copy()
    operation_ids = processed_df["Operation_Unique_Identifier"]
    processed_df["record_id"] = operation_ids.where(
        operation_ids.str.len().gt(0), processed_df.index.astype(str)
    )
    processed_df["CountryCode"] = processed_df["CountryCode"].map(
        lambda value: clean_text(value).upper()
    )

    for column in ["Country", "NUTS3_Label", "LAU_Labels"]:
        processed_df[column] = processed_df[column].map(clean_label)

    for column in ["Total_Eligible_Expenditure_amount", "Project_EU_Budget"]:
        processed_df[column] = pd.to_numeric(
            # remove spaces and comma thousands separators before numeric parsing
            processed_df[column].astype("string").str.replace(r"[\s,]", "", regex=True),
            errors="coerce",
        )

    coordinates = (
        processed_df["Location_Indicator_latitude_longitude"]
        .str.split("|", n=1)
        .str[0]
        # capture latitude and longitude from the first "lat, lon" pair
        .str.extract(r"^\s*([^,]+)\s*,\s*([^,]+)\s*$")
    )
    processed_df["latitude"] = pd.to_numeric(coordinates[0], errors="coerce")
    processed_df["longitude"] = pd.to_numeric(coordinates[1], errors="coerce")

    text_columns = [
        "Operation_Name_English",
        "Operation_Summary_English",
        "Operation_Name_Programme_Language",
        "Operation_Summary_Programme_Language",
    ]
    processed_df["search_text"] = processed_df[text_columns].agg(
        lambda row: clean_text(". ".join(value for value in row if value)), axis=1
    )
    processed_df = processed_df[processed_df["search_text"].str.len() > 0].reset_index(drop=True)
    processed_df["dataset_version"] = DATASET_VERSION
    processed_df = processed_df[PROCESSED_COLUMNS]
    processed_df.to_parquet(processed_path, index=False)

print(f"Total records: {len(processed_df)}, Unique countries: {processed_df['Country'].nunique()}")
display(processed_df.head(1))

Total records: 124988, Unique countries: 26


,Operation_Unique_Identifier,Operation_Local_Identifier,Operation_Name_English,Operation_Name_Programme_Language,Country,Operation_Start_Date,Operation_End_Date,Cofinancing_Rate,Total_Eligible_Expenditure_amount,Total_Eligible_Expenditure_Currency,...,Image_URL,InfoRegio_ID,InfoRegio_URL,CountryCode,source_file,record_id,latitude,longitude,search_text,dataset_version
0,https://linkedopendata.eu/entity/Q7500632,2021AT05FFPR001_51,Basic education at the Volkshochschule Salzburg,Basisbildung an der Volkshochschule Salzburg,Austria,01/07/2023,30/06/2025,40.0,144920.0,EUR,...,NaN,NaN,NaN,AT,latest_AT-pp21-27-latest.csv,https://linkedopendata.eu/entity/Q7500632,47.80948,13.030657,"Basic education at the Volkshochschule Salzburg. The basic education courses are aimed at all people with basic education needs (educationally disadvantaged, low-skilled people; persons with a migrant background and persons at risk of marginalisa...",v1


## BM25 lexical text

Creates the sparse retrieval view for BM25. It removes multilingual stop words, tokenizes the combined title and summary text, and stores the tokens.

The BM25 text is shorter than the readable text on average (1635.17 vs 2159.61 characters), so stop-word removal and token filtering are cutting noise before keyword scoring.

In [4]:
EU_LANGUAGE_CODES = [
    "bg",
    "cs",
    "da",
    "de",
    "el",
    "en",
    "es",
    "et",
    "fi",
    "fr",
    "hr",
    "hu",
    "it",
    "lt",
    "lv",
    "mt",
    "nl",
    "pl",
    "pt",
    "ro",
    "sk",
    "sl",
    "sv",
]

STOP_WORDS = set()
for language_code in EU_LANGUAGE_CODES:
    STOP_WORDS.update(stopwords.stopwords(language_code))


def tokenize(text):
    # match word tokens, including accented letters
    tokens = re.findall(r"[\wÀ-ÿ]+", clean_text(text).lower())
    return [token for token in tokens if token not in STOP_WORDS and len(token) > 1]


processed_df["bm25_tokens"] = processed_df["search_text"].map(tokenize)
processed_df["bm25_text"] = processed_df["bm25_tokens"].map(" ".join)

print(
    f"Average search_text length: {processed_df['search_text'].str.len().mean():.2f}, "
    f"Average bm25_text length: {processed_df['bm25_text'].str.len().mean():.2f}"
)

display(processed_df[["search_text", "bm25_text"]].head())

Average search_text length: 2159.61, Average bm25_text length: 1635.17


,search_text,bm25_text
0,"Basic education at the Volkshochschule Salzburg. The basic education courses are aimed at all people with basic education needs (educationally disadvantaged, low-skilled people; persons with a migrant background and persons at risk of marginalisa...",basic education volkshochschule salzburg basic education courses aimed people basic education educationally disadvantaged skilled people persons migrant background persons risk marginalisation socially regionally disadvantaged persons entry femal...
1,"Spielberg production campus. ecomaster technology gmbh is building a new plant at the Spielberg site (consisting of a hall and an office building) and will start the production of transfer stations, transformer stations, district storage and swit...",spielberg production campus ecomaster technology gmbh building plant spielberg consisting hall office building start production transfer stations transformer stations district storage switchgear power distribution cabinets produktionscampus spiel...
2,"JTF creates future space Salzkammergut Nord. Enable more companies in the region to implement research and innovation projects. In addition to regional needs, the focus is on digital and green transformation. Another important aspect is the possi...",jtf creates future space salzkammergut nord enable companies region implement innovation projects addition regional focus digital green transformation aspect possibilities artificial intelligence support companies unlocking existing potential res...
3,Continue learning with the School Success Association. continue learning with the school success association VEREIN SCHULERFOLGMag. DDr. Stefan UnterbergerOur goal is to make parents competent companions of their children in school issues and lea...,continue learning school success association continue learning school success association verein schulerfolgmag ddr stefan unterbergerour goal parents competent companions children school issues learning aware family origin influence child educat...
4,Computer courses 2024 in Güssing. The training measures for unemployed persons in the field of IT are indexed in terms of labour market policy and will be implemented in 2024. Participants should enter the IT course via an information day. Based ...,courses 2024 güssing training measures unemployed persons field indexed terms labour market policy implemented 2024 participants enter day based testing individual training plan created support plan tailored participant start day potential partic...


## BM25 index

Builds and caches the BM25 index used for keyword retrieval.

In [5]:
def corpus_texts_hash(texts):
    payload = json.dumps(texts, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def min_max_scale(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    return np.nan_to_num(minmax_scale(values))


# https://bm25s.github.io/
BM25_K1 = 1.5
BM25_B = 0.75
BM25_METHOD = "atire"
BM25_IDF_METHOD = "lucene"


def build_bm25_index(documents):
    index = bm25s.BM25(k1=BM25_K1, b=BM25_B, method=BM25_METHOD, idf_method=BM25_IDF_METHOD)
    index.index(documents, show_progress=True)
    return index


def bm25_scores(query):
    query_tokens = tokenize(query)
    if not query_tokens:
        return np.zeros(len(processed_df))
    index = bm25_index
    if index is None:
        raise RuntimeError("BM25 index does not exist")
    return index.get_scores(query_tokens).astype(float, copy=False)


BM25_INDEX_PATH = INDEX_DIR / f"bm25s_{DATASET_VERSION}.pkl"
bm25_documents = processed_df["bm25_tokens"].tolist()

if BM25_INDEX_PATH.exists():
    with open(BM25_INDEX_PATH, "rb") as file:
        bm25_index = pickle.load(file)

    print(f"BM25 index loaded from {BM25_INDEX_PATH}")
else:
    bm25_index = build_bm25_index(bm25_documents)

    with open(BM25_INDEX_PATH, "wb") as file:
        pickle.dump(bm25_index, file)

    print(f"BM25 index saved to {BM25_INDEX_PATH}")

BM25 index loaded from /Users/riccardo/Projects/luiss-xai/data/indexes/bm25s_v1.pkl


## Dense indexes

Builds the dense retrieval indexes. Each project text becomes an embedding vector, so the model can match projects with similar meaning even when the wording differs.

MPNet is the SBERT semantic baseline. Multilingual E5 checks whether retrieval works better when project descriptions use both English and programme-language text. The embeddings are cached so later runs do not need to rebuild them.

In [6]:
DENSE_CANDIDATES = {
    # https://huggingface.co/sentence-transformers/all-mpnet-base-v2
    "sbert_mpnet": "sentence-transformers/all-mpnet-base-v2",
    # https://huggingface.co/intfloat/multilingual-e5-large
    "multilingual_e5": "intfloat/multilingual-e5-large",
}
# this shows as the best dense candidate
DEFAULT_DENSE_CANDIDATE = "multilingual_e5"


def dense_texts_for_model(texts, candidate_name):
    # E5 uses explicit input prefixes; the other dense model uses raw text
    if candidate_name == "multilingual_e5":
        return [f"passage: {text}" for text in texts]
    return texts


def dense_query_for_model(query, candidate_name):
    if candidate_name == "multilingual_e5":
        return f"query: {query}"
    return query


def dense_cache_path(candidate_name):
    return INDEX_DIR / f"{candidate_name}_{DATASET_VERSION}.pkl"


def load_cached_embeddings(candidate_name, model_name, texts_hash):
    cache_path = dense_cache_path(candidate_name)
    if not cache_path.exists():
        return None

    with open(cache_path, "rb") as file:
        cached_index = pickle.load(file)

    if cached_index.get("model_name") != model_name:
        print(f"Ignoring dense cache for {candidate_name}: model changed.")
        return None
    if cached_index.get("dataset_version") != DATASET_VERSION:
        print(f"Ignoring dense cache for {candidate_name}: dataset version changed.")
        return None
    if cached_index.get("texts_hash") != texts_hash:
        print(f"Ignoring dense cache for {candidate_name}: corpus text changed.")
        return None

    print(f"Loaded cached embeddings for {candidate_name} from {cache_path}.")
    return cached_index["embeddings"]


def save_cached_embeddings(candidate_name, model_name, texts_hash, embeddings):
    cache_path = dense_cache_path(candidate_name)
    with open(cache_path, "wb") as file:
        pickle.dump(
            {
                "name": candidate_name,
                "model_name": model_name,
                "dataset_version": DATASET_VERSION,
                "texts_hash": texts_hash,
                "embeddings": embeddings,
            },
            file,
        )
    print(f"Saved embeddings for {candidate_name} to {cache_path}.")


def build_dense_index(candidate_name, model_name, texts):
    texts_hash = corpus_texts_hash(texts)
    embeddings = load_cached_embeddings(candidate_name, model_name, texts_hash)
    model = SentenceTransformer(model_name)

    if embeddings is None:
        embeddings = model.encode(
            dense_texts_for_model(texts, candidate_name),
            batch_size=32,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        save_cached_embeddings(candidate_name, model_name, texts_hash, embeddings)

    return {
        "name": candidate_name,
        "model_name": model_name,
        "model": model,
        "embeddings": embeddings,
    }


corpus_texts = processed_df["search_text"].tolist()
dense_indexes = {}

for candidate_name, model_name in DENSE_CANDIDATES.items():
    dense_indexes[candidate_name] = build_dense_index(candidate_name, model_name, corpus_texts)

Loaded cached embeddings for sbert_mpnet from /Users/riccardo/Projects/luiss-xai/data/indexes/sbert_mpnet_v1.pkl.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded cached embeddings for multilingual_e5 from /Users/riccardo/Projects/luiss-xai/data/indexes/multilingual_e5_v1.pkl.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Geocoding

Defines the geocoding step used when a location is available. Nominatim converts free-text locations into country and coordinate metadata; the notebook then turns country matches and distance into a geography score.

The example at the bottom shows the expected input and output for one location.

In [7]:
NOMINATIM_SEARCH_URL = "https://nominatim.openstreetmap.org/search"
NOMINATIM_USER_AGENT = "luiss-xai-geocoder/1.0"
NOMINATIM_EMAIL = os.getenv("NOMINATIM_EMAIL")
_last_nominatim_request_at = 0.0


# https://nominatim.org/
@lru_cache(maxsize=256)
def search_nominatim(query):
    global _last_nominatim_request_at

    params = {
        "q": query,
        "format": "jsonv2",
        "addressdetails": 1,
        "limit": 1,
        "accept-language": "en",
    }
    if NOMINATIM_EMAIL:
        params["email"] = NOMINATIM_EMAIL

    # normatim requires at most 1 request per second
    elapsed = time.monotonic() - _last_nominatim_request_at
    if elapsed < 1:
        time.sleep(1 - elapsed)

    url = f"{NOMINATIM_SEARCH_URL}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": NOMINATIM_USER_AGENT})
    _last_nominatim_request_at = time.monotonic()

    try:
        with urlopen(request, timeout=10) as response:
            data = json.load(response)
            return data if isinstance(data, list) else []
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError):
        return []


def geocode_location(location_text):
    query = clean_text(location_text)
    if not query:
        return {}

    metadata = {"raw_location": location_text}
    # treat two-letter alphabetic inputs as country codes
    # this lets us avoid unnecessary API calls
    if re.fullmatch(r"[A-Za-z]{2}", query):
        metadata["country_code"] = query.upper()
        return metadata

    results = search_nominatim(query)
    if not results:
        return metadata

    result = results[0]
    try:
        metadata["latitude"] = float(result["lat"])
        metadata["longitude"] = float(result["lon"])
    except (KeyError, TypeError, ValueError):
        return metadata

    address = result.get("address", {})
    country_code = address.get("country_code")
    if country_code:
        metadata["country_code"] = country_code.upper()
    if result.get("display_name"):
        metadata["display_name"] = result["display_name"]
    return metadata


EARTH_RADIUS_KM = 6371.0
# we give more weight to country matches since the dataset is very geographically sparse
GEOGRAPHY_COUNTRY_MATCH_WEIGHT = 0.7
# distance decay is steep because we want to prioritize very local matches
GEOGRAPHY_DISTANCE_WEIGHT = 0.3
GEOGRAPHY_DISTANCE_DECAY_KM = 500
GEOGRAPHY_COORDINATE_COLUMNS = ["latitude", "longitude"]


def geography_scores(location_metadata, records):
    if not location_metadata:
        return np.zeros(len(records))
    scores = np.zeros(len(records))
    query_country = location_metadata.get("country_code")
    if query_country:
        scores += (records["CountryCode"].fillna("").str.upper() == query_country).astype(
            float
        ) * GEOGRAPHY_COUNTRY_MATCH_WEIGHT
    coordinate_columns = set(GEOGRAPHY_COORDINATE_COLUMNS)
    has_query_coordinates = coordinate_columns.issubset(location_metadata)
    has_record_coordinates = coordinate_columns.issubset(records.columns)
    if has_query_coordinates and has_record_coordinates:
        record_coordinates = records[GEOGRAPHY_COORDINATE_COLUMNS].apply(
            pd.to_numeric, errors="coerce"
        )
        valid_coordinates = record_coordinates.notna().all(axis=1).to_numpy()
        distance_scores = np.zeros(len(records))

        if valid_coordinates.any():
            query_coordinates = np.radians(
                [[location_metadata["latitude"], location_metadata["longitude"]]]
            )
            candidate_coordinates = np.radians(
                record_coordinates.loc[valid_coordinates].to_numpy(dtype=float)
            )
            distances = (
                haversine_distances(query_coordinates, candidate_coordinates)[0] * EARTH_RADIUS_KM
            )
            distance_scores[valid_coordinates] = np.clip(
                1 - distances / GEOGRAPHY_DISTANCE_DECAY_KM, 0, 1
            )
        scores += distance_scores * GEOGRAPHY_DISTANCE_WEIGHT
    return np.clip(scores, 0, 1)


geocode_location("LUISS Viale Romania, Rome, Italy")

{'raw_location': 'LUISS Viale Romania, Rome, Italy',
 'latitude': 41.9244494,
 'longitude': 12.4941045,
 'country_code': 'IT',
 'display_name': 'Libera Università Internazionale degli Studi Sociali (LUISS) Guido Carli, Viale Romania, Parioli, Municipio Roma II, Rome, Roma Capitale, Lazio, 00197, Italy'}

## Hybrid model

Defines the retrieval function used by the prototype. It combines dense similarity, BM25 keyword evidence, and geography into one confidence score, then returns ranked projects with metadata, component scores, and searchable text for explanations.

The example query looks for past projects similar to turning an empty train station near Trento into a civic hub for climate work. The top result is a Rovereto project with confidence close to 1.0 because both the theme and location match closely.

In [8]:
CONFIDENCE_WEIGHT_CANDIDATES = [
    {"semantic": 0.70, "keyword": 0.20, "geographic": 0.10},
    {"semantic": 0.60, "keyword": 0.30, "geographic": 0.10},
    {"semantic": 0.60, "keyword": 0.25, "geographic": 0.15},
    {"semantic": 0.50, "keyword": 0.35, "geographic": 0.15},
    {"semantic": 0.50, "keyword": 0.30, "geographic": 0.20},
]
# this shows as the best confidence weighting candidate
DEFAULT_CONFIDENCE_WEIGHTS = CONFIDENCE_WEIGHT_CANDIDATES[3]
RESULT_DISPLAY_COLUMNS = ["rank", "confidence", "Country", "Fund_Name"]


def dense_scores(query, dense_index):
    candidate_name = dense_index["name"]
    query_embedding = dense_index["model"].encode(
        [dense_query_for_model(query, candidate_name)],
        normalize_embeddings=True,
    )
    return linear_kernel(dense_index["embeddings"], query_embedding).ravel()


def confidence_scores(semantic_score, keyword_score, geographic_score=None, weights=None):
    weights = DEFAULT_CONFIDENCE_WEIGHTS if weights is None else weights
    weighted_score = semantic_score * weights["semantic"] + keyword_score * weights["keyword"]
    total_weight = weights["semantic"] + weights["keyword"]

    if geographic_score is not None:
        weighted_score += geographic_score * weights["geographic"]
        total_weight += weights["geographic"]

    return weighted_score / total_weight


def retrieve_similar_projects(
    query,
    location_text="",
    top_k=TOP_K,
    dense_candidate=DEFAULT_DENSE_CANDIDATE,
    confidence_weights=None,
    exclusion_mask=None,
):
    location_metadata = geocode_location(location_text)
    keyword_score = min_max_scale(bm25_scores(query))

    selected_dense_index = dense_indexes[dense_candidate]
    semantic_score = min_max_scale(dense_scores(query, selected_dense_index))
    geographic_score = (
        geography_scores(location_metadata, processed_df) if location_metadata else None
    )
    confidence = confidence_scores(
        semantic_score, keyword_score, geographic_score, weights=confidence_weights
    )
    if exclusion_mask is not None:
        confidence = confidence.copy()
        confidence[np.asarray(exclusion_mask, dtype=bool)] = -np.inf

    ranked_indices = np.argsort(confidence)[::-1][:top_k]
    result_columns = [
        "Operation_Unique_Identifier",
        "Programme_Name",
        "Fund_Name",
        "Category_Label",
        "Specific_Objective_Label",
        "Policy_Objective_Label",
        "Total_Eligible_Expenditure_amount",
        "Project_EU_Budget",
        "Country",
        "NUTS3_Label",
        "LAU_Labels",
    ]
    results = processed_df.iloc[ranked_indices][result_columns + ["search_text"]].copy()

    results["semantic_score"] = semantic_score[ranked_indices]
    results["keyword_score"] = keyword_score[ranked_indices]
    results["geographic_score"] = (
        geographic_score[ranked_indices] if geographic_score is not None else np.nan
    )
    results["confidence"] = confidence[ranked_indices]
    results["rank"] = range(1, len(results) + 1)

    return results[
        ["rank", "confidence", "semantic_score", "keyword_score", "geographic_score"]
        + result_columns
        + ["search_text"]
    ]


def result_display_table(results):
    display_df = results[RESULT_DISPLAY_COLUMNS].copy()
    display_df["confidence"] = display_df["confidence"].round(3)
    display_df["project_text"] = results["search_text"].copy()
    return display_df


example_query = "Transform an empty train station into a public civic hub for climate change"
example_results = retrieve_similar_projects(example_query, location_text="Trento, Italy", top_k=5)
result_display_table(example_results)

,rank,confidence,Country,Fund_Name,project_text
83126,1,0.998,Italy,European Regional Development Fund,"S4T. The alpine town of Rovereto and its functional urban area face the challenge of rapidly adapting to the effects of climate change and effectively mitigating the resulting loss of biodiversity, which is closely linked to the territorial cultu..."
62714,2,0.559,France,European Regional Development Fund,"Creation of a multimodal exchange hub at Hazebrouck station. Coeur de Flandre agglo is the contracting authority for the creation of a multimodal exchange hub at Hazebrouck station. With more than 6,500 descents per day, the intercommunality want..."
277,3,0.531,Austria,Just Transition Fund,"Transform JTF region (Kirchdorf, Wels, Wels-Land): Climate neutrality through innovation. The project aims to strengthen the JTF region of Kirchdorf, Wels-Land and Wels Stadt through targeted measures in the areas of innovation, sustainability an..."
83105,4,0.527,Ireland,Just Transition Fund,"Ballaghaderreen Just Transition Hub. Located on the Sligo road in Ballaghaderreen, County Roscommon, this partially completed filling station was recently purchased by Roscommon County Council along with 13 derelict houses in Shannon Valley, Ball..."
60166,5,0.503,France,European Regional Development Fund,"Climate Change Adaptation Ambassadors. The project aims to train young people between the ages of 18 and 25 in the concept of adapting to climate change and its challenges. Trained, these young people, welcomed into civic service by the beneficia..."


## Benchmark queries

Creates 200 benchmark queries from historical project records. The query text comes from the project description, but label names are removed so the model cannot match the answer directly.

The original programme, fund, category, and objective labels are kept separately for relevance scoring. The source record is also removed from its own retrieval results, so the evaluation does not reward the model for returning the same project again.

The benchmark file is written under `data/benchmarks/`; the sample at the bottom shows the resulting fields.

In [9]:
BENCHMARK_SIZE = min(200, len(processed_df))
# we sample from longer search_text entries to ensure the benchmark queries are sufficiently complex
benchmark_candidates = processed_df[processed_df["search_text"].str.len().gt(80)].sample(
    n=BENCHMARK_SIZE, random_state=RANDOM_SEED
)


# to create more challenging benchmark queries, we remove
# any label text from the search_text that could give away the answer
def strip_label_clues(text, row):
    stripped_text = clean_text(text)
    label_fields = [
        "Programme_Name",
        "Fund_Name",
        "Category_Label",
        "Specific_Objective_Label",
        "Policy_Objective_Label",
        "Country",
        "NUTS3_Label",
        "LAU_Labels",
    ]

    for field in label_fields:
        value = clean_text(row.get(field))
        if value:
            # remove the exact label text ignoring case
            stripped_text = re.sub(re.escape(value), " ", stripped_text, flags=re.IGNORECASE)

    return clean_text(stripped_text)


benchmark_df = benchmark_candidates[
    [
        "record_id",
        "CountryCode",
        "Programme_Name",
        "Fund_Name",
        "Category_Label",
        "Specific_Objective_Label",
        "Policy_Objective_Label",
    ]
].rename(columns={"record_id": "source_record_id"})
benchmark_df["query_text"] = benchmark_candidates.apply(
    lambda row: strip_label_clues(row["search_text"], row), axis=1
)
benchmark_df = benchmark_df[
    ["source_record_id", "query_text"]
    + [
        column
        for column in benchmark_df.columns
        if column not in ["source_record_id", "query_text"]
    ]
]
benchmark_ids = set(benchmark_df["source_record_id"])
benchmark_exclusion_mask = processed_df["record_id"].isin(benchmark_ids).to_numpy()

benchmark_path = BENCHMARK_DIR / f"benchmark_queries_{DATASET_VERSION}.csv"
benchmark_df.to_csv(benchmark_path, index=False)
display(benchmark_df.head(1))

,source_record_id,query_text,CountryCode,Programme_Name,Fund_Name,Category_Label,Specific_Objective_Label,Policy_Objective_Label
109264,https://linkedopendata.eu/entity/Q7462945,"FMC: Lifelong learning and employability. The central purpose of the operation is to respond to the training needs of adults, to strengthen the skills of participants and to promote employability. Particular emphasis will be placed on the Digital...",PT,"Demography, Qualifications and Inclusion – PT – ESF+",European Social Fund Plus,Support for adult education (excluding infrastructure),ESO4.7,Social Europe


## Label distribution

Checks how broad or narrow the relevance labels are. Retrieved projects get higher grades when they share a specific objective or category with the benchmark query, lower grades when they only share broader labels such as policy objective, fund, or programme, and zero when none of those fields match.

The first table shows why this matters. `Fund_Name` has 7 unique values and its most common value covers about 50% of the dataset. `Policy_Objective_Label` has 10 unique values and its most common value covers about 45%. Those fields give context, but they are too broad to prove a close match by themselves.

`Category_Label` is more specific, with 619 unique values and a top-value share of about 14%. `Specific_Objective_Label` has 65 unique values, so it sits between broad policy labels and detailed category labels.

The second table shows the same pattern through match-pool sizes. Fund and policy labels create very large pools, while category and programme labels produce smaller pools. Category and specific-objective matches are therefore stronger evidence in the benchmark.

In [10]:
RELEVANCE_SIGNAL_FIELDS = [
    "Specific_Objective_Label",
    "Category_Label",
    "Policy_Objective_Label",
    "Fund_Name",
    "Programme_Name",
]


def label_distribution_summary(records, fields):
    rows = []
    for field in fields:
        values = records[field].fillna("").map(clean_text)
        values = values[values.str.len().gt(0)]
        counts = values.value_counts()
        rows.append(
            {
                "field": field,
                "unique_values": len(counts),
                "top_value_share": int(counts.iloc[0]) / len(records) if len(counts) else 0,
                "top_10_share": counts.head(10).sum() / len(records),
            }
        )
    return pd.DataFrame(rows)


def benchmark_match_pool_summary(records, benchmark_queries, fields):
    rows = []
    for field in fields:
        values = records[field].fillna("").map(clean_text)
        pool_sizes = []
        for _, query_row in benchmark_queries.iterrows():
            value = clean_text(query_row.get(field))
            pool_sizes.append(int((values == value).sum()) if value else 0)
        pool_sizes = pd.Series(pool_sizes)
        rows.append(
            {
                "field": field,
                "mean_pool_size": pool_sizes.mean(),
                "median_pool_size": pool_sizes.median(),
                "p90_pool_size": pool_sizes.quantile(0.90),
            }
        )
    return pd.DataFrame(rows)


def same_label(query_row, result_row, field):
    query_value = clean_text(query_row.get(field))
    return bool(query_value) and query_value == clean_text(result_row.get(field))


def relevance_grade(query_row, result_row):
    same_specific_objective = same_label(query_row, result_row, "Specific_Objective_Label")
    same_category = same_label(query_row, result_row, "Category_Label")
    same_policy_objective = same_label(query_row, result_row, "Policy_Objective_Label")
    same_fund = same_label(query_row, result_row, "Fund_Name")
    same_programme = same_label(query_row, result_row, "Programme_Name")

    if same_specific_objective and same_category:
        return 3
    if same_specific_objective or same_category:
        return 2
    if same_policy_objective or same_fund or same_programme:
        return 1
    return 0


relevance_label_distribution = label_distribution_summary(processed_df, RELEVANCE_SIGNAL_FIELDS)
# the heldout corpus consists of all records
# that are not part of the benchmark queries
# else the evaluation would be trivial since the exact same record would be in the corpus
benchmark_match_pool_distribution = benchmark_match_pool_summary(
    processed_df[~benchmark_exclusion_mask], benchmark_df, RELEVANCE_SIGNAL_FIELDS
)

display(relevance_label_distribution)
display(benchmark_match_pool_distribution)

,field,unique_values,top_value_share,top_10_share
0,Specific_Objective_Label,65,0.187666,0.714517
1,Category_Label,619,0.135325,0.447243
2,Policy_Objective_Label,10,0.450651,0.998800
3,Fund_Name,7,0.500696,1.000000
4,Programme_Name,173,0.103226,0.494535


,field,mean_pool_size,median_pool_size,p90_pool_size
0,Specific_Objective_Label,9396.820,6607.5,23415.0
1,Category_Label,4550.715,2331.0,16883.0
2,Policy_Objective_Label,39403.795,39602.0,56240.0
3,Fund_Name,53129.580,57431.0,62481.0
4,Programme_Name,5347.695,3725.0,12876.0


## Benchmark results

Runs the 200 benchmark queries against each dense retriever and confidence-weight combination. The comparison uses Precision@k, Mean Reciprocal Rank, and NDCG@k.

`multilingual_e5` performs best overall. Its best configuration ranks first by NDCG@10 with weights 0.50 semantic, 0.35 keyword, and 0.15 geography.

In [11]:
def ndcg_at_k(grades, k):
    gains = np.asarray([2**grade - 1 for grade in grades[:k]], dtype=float)
    if gains.size == 0 or np.all(gains == 0):
        return 0.0
    if gains.size == 1:
        return 1.0
    ranking_scores = np.arange(len(gains), 0, -1)
    return float(ndcg_score(np.asarray([gains]), np.asarray([ranking_scores]), k=k))


def confidence_weights_label(weights):
    return f"{weights['semantic']:.2f}/{weights['keyword']:.2f}/{weights['geographic']:.2f}"


def normalize_confidence_weights_label(label):
    numbers = re.findall(r"\d+\.\d+", str(label))
    return "/".join(numbers[:3]) if len(numbers) >= 3 else label


def evaluate_retriever_diagnostics(
    benchmark_queries,
    top_k=TOP_K,
    dense_candidate=DEFAULT_DENSE_CANDIDATE,
    confidence_weights=None,
    exclusion_mask=None,
):
    weights = confidence_weights or DEFAULT_CONFIDENCE_WEIGHTS
    diagnostic_rows = []

    for _, query_row in benchmark_queries.iterrows():
        results = retrieve_similar_projects(
            query_row["query_text"],
            location_text=query_row.get("CountryCode", ""),
            top_k=top_k,
            dense_candidate=dense_candidate,
            confidence_weights=weights,
            exclusion_mask=exclusion_mask,
        )
        grades = [relevance_grade(query_row, result_row) for _, result_row in results.iterrows()]
        relevant = [grade >= 2 for grade in grades]
        first_relevant_rank = next(
            (index + 1 for index, value in enumerate(relevant) if value), None
        )
        diagnostic_rows.append(
            {
                "candidate": dense_candidate,
                "confidence_weights": weights,
                "confidence_weights_label": confidence_weights_label(weights),
                "source_record_id": query_row["source_record_id"],
                "query_country": query_row.get("CountryCode", ""),
                "query_fund": query_row.get("Fund_Name", ""),
                "query_policy_objective": query_row.get("Policy_Objective_Label", ""),
                "query_specific_objective": query_row.get("Specific_Objective_Label", ""),
                "precision_at_k": sum(relevant) / top_k,
                "reciprocal_rank": 1 / first_relevant_rank if first_relevant_rank else 0.0,
                "first_relevant_rank": first_relevant_rank,
                "ndcg_at_k": ndcg_at_k(grades, top_k),
                "relevant_count": sum(relevant),
                "top_result_ids": results["Operation_Unique_Identifier"].tolist(),
                "top_result_grades": grades,
                "top_result_confidences": results["confidence"].round(6).tolist(),
                "top_k": top_k,
            }
        )

    return pd.DataFrame(diagnostic_rows)


def summarize_evaluation_diagnostics(evaluation_diagnostics):
    summary = (
        evaluation_diagnostics.groupby(["candidate", "confidence_weights_label"], as_index=False)
        .agg(
            confidence_weights=("confidence_weights", "first"),
            precision_at_k=("precision_at_k", "mean"),
            mrr=("reciprocal_rank", "mean"),
            ndcg_at_k=("ndcg_at_k", "mean"),
        )
        .sort_values("ndcg_at_k", ascending=False, ignore_index=True)
    )
    return summary[
        [
            "candidate",
            "confidence_weights",
            "confidence_weights_label",
            "precision_at_k",
            "mrr",
            "ndcg_at_k",
        ]
    ]


def evaluate_weight_grid_diagnostics(
    benchmark_queries,
    top_k=TOP_K,
    dense_candidates=None,
    exclusion_mask=None,
):
    dense_candidates = list(DENSE_CANDIDATES) if dense_candidates is None else dense_candidates
    return pd.concat(
        [
            evaluate_retriever_diagnostics(
                benchmark_queries,
                top_k=top_k,
                dense_candidate=dense_candidate,
                confidence_weights=weights,
                exclusion_mask=exclusion_mask,
            )
            for dense_candidate in dense_candidates
            for weights in CONFIDENCE_WEIGHT_CANDIDATES
        ],
        ignore_index=True,
    )


benchmark_evaluation = (
    BENCHMARK_DIR / f"benchmark_queries_evaluation_{DATASET_VERSION}_top{TOP_K}.csv"
)

if benchmark_evaluation.exists():
    evaluation_diagnostics = pd.read_csv(benchmark_evaluation)
    evaluation_diagnostics["confidence_weights_label"] = evaluation_diagnostics[
        "confidence_weights_label"
    ].map(normalize_confidence_weights_label)
    print(f"Loaded existing evaluation from {benchmark_evaluation}.")
else:
    evaluation_diagnostics = evaluate_weight_grid_diagnostics(
        benchmark_df,
        top_k=TOP_K,
        exclusion_mask=benchmark_exclusion_mask,
    )
    evaluation_diagnostics.to_csv(benchmark_evaluation, index=False)

evaluation_summary = summarize_evaluation_diagnostics(evaluation_diagnostics)
display(evaluation_summary)

Loaded existing evaluation from /Users/riccardo/Projects/luiss-xai/data/benchmarks/benchmark_queries_evaluation_v1_top10.csv.


,candidate,confidence_weights,confidence_weights_label,precision_at_k,mrr,ndcg_at_k
0,multilingual_e5,"{'geographic': 0.15, 'keyword': 0.35, 'semantic': 0.5}",0.50/0.35/0.15,0.8225,0.918472,0.940790
1,multilingual_e5,"{'geographic': 0.1, 'keyword': 0.3, 'semantic': 0.6}",0.60/0.30/0.10,0.8185,0.916556,0.940146
2,multilingual_e5,"{'geographic': 0.2, 'keyword': 0.3, 'semantic': 0.5}",0.50/0.30/0.20,0.8220,0.921548,0.939532
3,multilingual_e5,"{'geographic': 0.15, 'keyword': 0.25, 'semantic': 0.6}",0.60/0.25/0.15,0.8195,0.918881,0.938811
4,sbert_mpnet,"{'geographic': 0.15, 'keyword': 0.35, 'semantic': 0.5}",0.50/0.35/0.15,0.8210,0.929673,0.937318
5,sbert_mpnet,"{'geographic': 0.2, 'keyword': 0.3, 'semantic': 0.5}",0.50/0.30/0.20,0.8185,0.925556,0.935174
6,multilingual_e5,"{'geographic': 0.1, 'keyword': 0.2, 'semantic': 0.7}",0.70/0.20/0.10,0.8170,0.919067,0.933576
7,sbert_mpnet,"{'geographic': 0.15, 'keyword': 0.25, 'semantic': 0.6}",0.60/0.25/0.15,0.8240,0.918798,0.933141
8,sbert_mpnet,"{'geographic': 0.1, 'keyword': 0.2, 'semantic': 0.7}",0.70/0.20/0.10,0.8275,0.923853,0.932370
9,sbert_mpnet,"{'geographic': 0.1, 'keyword': 0.3, 'semantic': 0.6}",0.60/0.30/0.10,0.8250,0.912056,0.929797


## RAG

Builds the explanation layer for retrieved results. First, the notebook summarizes the top matches into programme, fund, category, objective, and budget suggestions. Then it sends each ranked match to the local Ollama model, `gemma4:e4b`, with the retrieved evidence and score components.

The example at the bottom shows the final output table: ranked matches, confidence scores, budget benchmark, explanations, and geography notes.

In [12]:
# https://ollama.com/library/gemma4:e4b
LOCAL_LLM_MODEL_VERSION = "gemma4:e4b"
OLLAMA_GENERATE_URL = "http://localhost:11434/api/generate"
LOCAL_LLM_TIMEOUT_SECONDS = 120
PROMPT_TEMPLATE_VERSION = "xai_v1"

POSITIONING_SUGGESTION_FIELDS = {
    "Programme_Name": "programme_name",
    "Fund_Name": "fund_name",
    "Category_Label": "category_label",
    "Specific_Objective_Label": "specific_objective_label",
    "Policy_Objective_Label": "policy_objective_label",
}
BUDGET_SUGGESTION_FIELDS = {
    "Total_Eligible_Expenditure_amount": "total_eligible_expenditure_amount",
    "Project_EU_Budget": "project_eu_budget",
}
MATCH_METADATA_FIELDS = POSITIONING_SUGGESTION_FIELDS | BUDGET_SUGGESTION_FIELDS

EXPLANATION_PROMPT_TEMPLATE = """
You are an explanation assistant for an EU funding retrieval system.

The ranking model has already selected this historical project. Only explain why it was matched.

Rules:
- Use only the provided JSON input.
- Do not change the project rank or confidence score.
- Do not invent programme, fund, category, objective, budget, or location information.
- Do not claim that the user project is eligible for funding.
- Explain the match in plain English for a non-technical user.
- Do not include internal reasoning steps or notes.
- If geography did not contribute to the score, say that geography was not used.
- Return valid JSON only, with no markdown.

Input JSON:
{{llm_input_json}}

Return this JSON structure:
{
  "project_id": "...",
  "rank": 1,
  "confidence_score": 0.0,
  "explanation": "...",
  "geography_note": "..."
}
"""


def json_ready_value(value):
    if value is None:
        return None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)
    if isinstance(value, float):
        return None if np.isnan(value) else value
    if pd.isna(value):
        return None
    if isinstance(value, str):
        text = clean_text(value)
        return text if text else None
    return value


def build_llm_input(input_text, location_text, result_row):
    search_text = clean_text(result_row.get("search_text", ""))
    title_summary_parts = search_text.split(". ", 1)
    title = title_summary_parts[0]
    summary = title_summary_parts[1] if len(title_summary_parts) > 1 else search_text

    return {
        "user_project": {
            "text": clean_text(input_text),
            "location": json_ready_value(location_text),
        },
        "matched_project": {
            "project_id": json_ready_value(result_row.get("Operation_Unique_Identifier")),
            "rank": int(result_row["rank"]),
            "title": title,
            "summary": summary,
            "country": json_ready_value(result_row.get("Country")),
            "nuts3_label": json_ready_value(result_row.get("NUTS3_Label")),
            "lau_labels": json_ready_value(result_row.get("LAU_Labels")),
        },
        "matched_project_metadata": {
            output_name: json_ready_value(result_row.get(source_name))
            for source_name, output_name in MATCH_METADATA_FIELDS.items()
        },
        "scores": {
            "semantic_score": json_ready_value(result_row.get("semantic_score")),
            "keyword_score": json_ready_value(result_row.get("keyword_score")),
            "geographic_score": json_ready_value(result_row.get("geographic_score")),
            "confidence_score": json_ready_value(result_row.get("confidence")),
        },
    }


def build_explanation_prompt(llm_input):
    llm_input_json = json.dumps(llm_input, indent=2, ensure_ascii=False)
    return EXPLANATION_PROMPT_TEMPLATE.replace("{{llm_input_json}}", llm_input_json)


def request_local_llm(prompt, model_name=LOCAL_LLM_MODEL_VERSION):
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {"temperature": 0},
    }
    request = Request(
        OLLAMA_GENERATE_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=LOCAL_LLM_TIMEOUT_SECONDS) as response:
            return json.load(response)
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as error:
        raise RuntimeError("Ollama request failed") from error


def validate_llm_output(llm_output, llm_input):
    required_fields = [
        "project_id",
        "rank",
        "confidence_score",
        "explanation",
        "geography_note",
    ]
    missing_fields = [field for field in required_fields if field not in llm_output]
    if missing_fields:
        raise ValueError(f"Local LLM output is missing fields: {', '.join(missing_fields)}")

    expected_project_id = llm_input["matched_project"]["project_id"]
    expected_rank = llm_input["matched_project"]["rank"]
    expected_confidence = llm_input["scores"]["confidence_score"]
    if (
        llm_output["project_id"] != expected_project_id
        or int(llm_output["rank"]) != expected_rank
        or not np.isclose(float(llm_output["confidence_score"]), float(expected_confidence))
    ):
        raise ValueError("Local LLM output has different project_id, rank or confidence_score")

    explanation = clean_text(llm_output["explanation"])
    geography_note = clean_text(llm_output["geography_note"])
    if not explanation or not geography_note:
        raise ValueError("Local LLM returned an empty explanation or geography_note")

    return {
        "project_id": expected_project_id,
        "rank": expected_rank,
        "confidence_score": expected_confidence,
        "explanation": explanation,
        "geography_note": geography_note,
    }


def generate_match_explanation(llm_input, model_name=LOCAL_LLM_MODEL_VERSION):
    prompt = build_explanation_prompt(llm_input)
    response = request_local_llm(prompt, model_name=model_name)
    try:
        llm_output = json.loads(response["response"])
    except (KeyError, TypeError, json.JSONDecodeError) as error:
        raise ValueError("Local LLM response does not match the expected JSON") from error
    return validate_llm_output(llm_output, llm_input)


def summarize_suggestions(input_text, location_text, results):
    positioning_suggestions = {}
    for source_name, output_name in POSITIONING_SUGGESTION_FIELDS.items():
        values = results[source_name].dropna().map(clean_text)
        values = values[values.str.len() > 0]
        positioning_suggestions[output_name] = [
            {"value": value, "match_count": int(count)}
            for value, count in values.value_counts().head(3).items()
        ]

    budget_benchmarks = {}
    for source_name, output_name in BUDGET_SUGGESTION_FIELDS.items():
        numeric_values = pd.to_numeric(results[source_name], errors="coerce").dropna()
        budget_benchmarks[output_name] = {
            "median": float(numeric_values.median()) if len(numeric_values) else None,
        }
    positioning_suggestions["budget_benchmarks"] = budget_benchmarks

    llm_inputs = [
        build_llm_input(input_text, location_text, result_row)
        for _, result_row in results.iterrows()
    ]
    llm_outputs = [generate_match_explanation(llm_input) for llm_input in llm_inputs]

    return {
        "positioning_suggestions": positioning_suggestions,
        "llm_inputs": llm_inputs,
        "llm_outputs": llm_outputs,
    }


def suggestion_summary_table(suggestion, row_count=TOP_K):
    ranked_inputs = sorted(
        suggestion["llm_inputs"],
        key=lambda llm_input: llm_input["scores"]["confidence_score"] or 0,
        reverse=True,
    )[:row_count]
    explanations_by_project_id = {
        output["project_id"]: output for output in suggestion["llm_outputs"]
    }
    budget_benchmark = suggestion["positioning_suggestions"]["budget_benchmarks"][
        "total_eligible_expenditure_amount"
    ]

    rows = []
    for llm_input in ranked_inputs:
        matched_project = llm_input["matched_project"]
        metadata = llm_input["matched_project_metadata"]
        scores = llm_input["scores"]
        explanation = explanations_by_project_id.get(matched_project["project_id"], {})
        rows.append(
            {
                "rank": matched_project["rank"],
                "title": matched_project["title"],
                "country": matched_project["country"],
                "fund": metadata["fund_name"],
                "confidence": scores["confidence_score"],
                "budget_median": budget_benchmark["median"],
                "explanation": explanation.get("explanation"),
                "geography_note": explanation.get("geography_note"),
            }
        )

    return pd.DataFrame(rows)


suggestions = summarize_suggestions(
    example_query,
    "Rovereto, Italy",
    example_results,
)

suggestion_table = suggestion_summary_table(suggestions)
display(suggestion_table)

,rank,title,country,fund,confidence,budget_median,explanation,geography_note
0,1,S4T,Italy,European Regional Development Fund,0.997550,1633333.33,"The match was made because both projects focus on transforming the empty train station in Rovereto into a public civic hub. Both also address major themes such as climate change, biodiversity loss, and regeneration of the local cultural heritage.","The location, Rovereto, Italy, contributed to the match score."
1,2,Creation of a multimodal exchange hub at Hazebrouck station,France,European Regional Development Fund,0.558637,1633333.33,"The user project and the matched project both focus on transforming a station area into a central hub. The matched project describes the creation of a multimodal exchange hub at Hazebrouck station, which includes elements like parking, bus statio...",Geography was not used.
2,3,"Transform JTF region (Kirchdorf, Wels, Wels-Land): Climate neutrality through innovation",Austria,Just Transition Fund,0.530908,1633333.33,"The project was matched because both projects focus on transforming physical spaces and regions to promote sustainability, innovation, and climate neutrality. The matched project aims to strengthen a region by establishing an innovation hub and p...","The match considered the location, but it was not the primary factor in the score."
3,4,Ballaghaderreen Just Transition Hub,Ireland,Just Transition Fund,0.526845,1633333.33,"The match was made because both projects involve transforming an abandoned or underutilized building into a community hub. The user project focuses on transforming a train station, while the matched project aims to transform a long abandoned fill...",Geography was not used.
4,5,Climate Change Adaptation Ambassadors,France,European Regional Development Fund,0.503211,1633333.33,"The project was matched because both the user project and the matched project focus on climate change adaptation. The user project mentions transforming a station for climate change, and the matched project aims to train young people to raise awa...",Geography was not used.


## Audit block

Records each run in an audit log so the retrieval and explanation process can be checked later. The block stores hashes of the input text, location metadata, retrieved result IDs, LLM inputs, explanations, and final suggestions. If any part of the run changes, the hash no longer matches.

Each block also stores the previous block hash, which chains the runs in order. The example at the bottom writes one block with the run ID, model version, dataset version, result IDs, previous block hash, and current block hash.

In [13]:
def stable_hash(value):
    payload = json.dumps(value, sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def latest_block_hash():
    blocks = sorted(AUDIT_DIR.glob("*.json"))
    if not blocks:
        return "0" * 64
    with open(blocks[-1], encoding="utf-8") as file:
        return json.load(file)["current_block_hash"]


def write_audit_block(input_text, location_metadata, results, suggestions):
    llm_inputs = suggestions.get("llm_inputs", [])
    llm_outputs = suggestions.get("llm_outputs", [])
    positioning_suggestions = suggestions.get("positioning_suggestions", suggestions)

    block_without_hash = {
        "run_id": str(uuid.uuid4()),
        "timestamp": datetime.now(UTC).isoformat(),
        "input_text_hash": stable_hash(clean_text(input_text)),
        "location_metadata_hash": (stable_hash(location_metadata) if location_metadata else None),
        "model_version": MODEL_VERSION,
        "dataset_version": DATASET_VERSION,
        "top_k_result_ids": results["Operation_Unique_Identifier"].tolist(),
        "llm_input_hash": stable_hash(llm_inputs) if llm_inputs else None,
        "prompt_template_version": PROMPT_TEMPLATE_VERSION if llm_inputs else None,
        "llm_model_version": LOCAL_LLM_MODEL_VERSION if llm_outputs else None,
        "explanation_hash": stable_hash(llm_outputs) if llm_outputs else None,
        "suggestions_hash": stable_hash(positioning_suggestions),
        "previous_block_hash": latest_block_hash(),
    }
    block = dict(block_without_hash)
    block["current_block_hash"] = stable_hash(block_without_hash)
    timestamp = str(block["timestamp"]).replace(":", "-")
    block_path = AUDIT_DIR / f"{timestamp}_{block['run_id']}.json"
    with open(block_path, "w", encoding="utf-8") as file:
        json.dump(block, file, indent=2, ensure_ascii=False)
    return block_path, block


audit_block = write_audit_block(
    example_query, geocode_location("Italy"), example_results, suggestions
)[1]
display(audit_block)

{'run_id': 'f71ec5f4-b1a8-491e-bbe5-c5f53eb97071',
 'timestamp': '2026-05-10T12:40:02.004923+00:00',
 'input_text_hash': '42e2d0cbd6a7e6f3ed325f4dbb8de68eb0d791a0733ff8a2fc9b4a6b7d4ba65b',
 'location_metadata_hash': 'e5eb35a7e657e340e0882183dce7502a0bd7beaadb9f44f3325d38c1019b6b1b',
 'model_version': 'hybrid_bm25_dense_geo_v1',
 'dataset_version': 'v1',
 'top_k_result_ids': ['https://linkedopendata.eu/entity/Q7420768',
  'https://linkedopendata.eu/entity/Q7357340',
  'https://linkedopendata.eu/entity/Q7501034',
  'https://linkedopendata.eu/entity/Q7361318',
  'https://linkedopendata.eu/entity/Q7512806'],
 'llm_input_hash': 'b4cc335c6665e42e9ab8247a214469f370d99f6c8610daf36ef6375648a1ae44',
 'prompt_template_version': 'xai_v1',
 'llm_model_version': 'gemma4:e4b',
 'explanation_hash': 'f4180ccced9dc5912a73fabbc7110400cf200e95e5ce1ec4abcce4cf3374434c',
 'suggestions_hash': '87c5ad5e41e13c7ef62aed0910e339d0e1c7f69c3d806127e2dda23551f7cca1',
 'previous_block_hash': '91494d3d300507d94cae3a48

## User example

Runs the full prototype on a sample idea: renovating public school buildings in Italy with insulation, smart energy monitoring, and renewable heating.

The top matches are all energy-efficiency renovation projects for public or school buildings. They mention insulation, heating-system replacement, renewable energy, and energy performance improvements. The retrieved projects come from Romania, Poland, France, Czechia, and Slovakia, and all top 10 matches are funded by the European Regional Development Fund.

Confidence ranges from about 0.75 for the top match to about 0.68 for the tenth. The budget benchmark is a median eligible expenditure of about 573.000 EUR. Geography was not used in the explanations because none of the retrieved projects are Italian records.

In [14]:
user_project_idea = """
A municipality wants to renovate public school buildings with better insulation,
smart energy monitoring, and renewable heating systems to lower emissions and operating costs.
"""
user_location = "Italy"

user_results = retrieve_similar_projects(
    user_project_idea, location_text=user_location, top_k=TOP_K
)
user_suggestions = summarize_suggestions(
    user_project_idea,
    user_location,
    user_results,
)
write_audit_block(
    user_project_idea,
    geocode_location(user_location),
    user_results,
    user_suggestions,
)

suggestion_summary_table(user_suggestions)

,rank,title,country,fund,confidence,budget_median,explanation,geography_note
0,1,"Increasing energy efficiency at Jidostița Secondary School, Com",Romania,European Regional Development Fund,0.754420,573674.96,"The project was matched because both the user's project and the selected project focus on improving the energy efficiency of public buildings. Specifically, both involve measures like improving energy performance, reducing thermal consumption, an...",Geography was not used.
1,2,Improving the energy efficiency of public buildings in the municipality of Sanok,Poland,European Regional Development Fund,0.736224,573674.96,"The project was matched because both the user's interest in renovating public school buildings with better insulation, smart energy monitoring, and renewable heating systems to lower emissions and operating costs, and the matched project's focus ...",Geography was not used.
2,3,Energy renovation of local public buildings,France,European Regional Development Fund,0.731922,573674.96,"The matched project is highly relevant because both projects focus on energy efficiency improvements for public school buildings. Specific measures mentioned in the matched project, such as exterior and interior insulation, LED lighting replaceme...",Geography was not used.
3,4,Complex austerity measures - OÚ Rychnovek,Czechia,European Regional Development Fund,0.704251,573674.96,"The matched project is relevant because both projects focus on improving public buildings through energy efficiency measures. Specifically, the matched project involves insulation, replacing heating sources, and installing photovoltaic power plan...",Geography was not used.
4,5,LOZORNO - BASIC SCHOOL - Improving the energy efficiency of public buildings,Slovakia,European Regional Development Fund,0.702707,573674.96,"The project is highly relevant because both the user project and the matched project focus on improving the energy performance and insulation of public school buildings. The matched project details specific activities like insulating the shell, r...",Geography was not used.
5,6,Building modifications of the elementary school Rychnovek - insulation of the building and related works,Czechia,European Regional Development Fund,0.700188,573674.96,"The project is highly relevant because both the user project and the matched project focus on energy efficiency improvements in public school buildings. Specifically, both involve insulation (of walls, roofs, and ceilings), replacing existing hea...",Geography was not used.
6,7,Improving the energy efficiency of public buildings in the Municipality of Zagórz,Poland,European Regional Development Fund,0.690226,573674.96,"The project focuses on improving the energy efficiency of public buildings, which aligns with your interest in renovating public school buildings with better insulation, smart energy monitoring, and renewable heating systems to lower emissions an...",Geography was not used.
7,8,Thermomodernization of selected public buildings in Legnica,Poland,European Regional Development Fund,0.689619,573674.96,"The project focuses on improving the energy efficiency of public buildings through thermomodernization. This aligns with the user's interest in renovating public school buildings with better insulation, smart energy monitoring, and renewable heat...",Geography was not used.
8,9,"Thermomodernisation of buildings: Primary School, Secondary School Complex and gymnasium in Sława with modernization of heat sources",Poland,European Regional Development Fund,0.686558,573674.96,"The project was matched because both the user project and the matched project focus on improving the energy efficiency of public buildings. Specifically, the matched project involves thermal modernization, insulation, and the replacement of heati...",Geography was not used.
9,10,Energy solution of VOŠ and SPŠE buildings,Czechia,European Regional Development Fund,0.684237,573674.96,"The matched project is highly relevant because 